# Module 9 — Audit metrics, code, and claims
## Worked examples for review

Use `module09_in_class.ipynb` to make predictions first. This notebook contains complete answers and runnable checks. Select this repo's `.venv` kernel after `uv sync --frozen`, then run from top to bottom. All data are invented teaching examples; no downloads or API keys are needed.

Define → predict → inspect → verify → record. Passing small checks supports a claim; it is not proof for every input.


In [1]:
import pandas as pd
import random
from itertools import permutations

print("Ready for the four examples.")

Ready for the four examples.


## 1 · Customer loyalty
**Request:** “Show which store has the most loyal customers.”

Before coding, ask: Who uses the result? What decision follows? Does loyalty mean frequent orders, repeat visits, spending, or something else?

For this example, all rows are distinct completed orders in January 2026. Customer IDs are reliable; every order belongs to one store. We compare repeat purchasing during this month, not long-run retention or a causal effect of the store.

In [2]:
orders = pd.DataFrame({
    "order": ["o1", "o2", "o3", "o4", "o5", "o6"],
    "store": ["A", "A", "A", "A", "B", "B"],
    "customer": ["Ana", "Ana", "Ana", "Ben", "Cai", "Cai"],
    "date": ["2026-01-02", "2026-01-03", "2026-01-04",
             "2026-01-02", "2026-01-02", "2026-01-05"]
})
print(orders.to_string(index=False))

order store customer       date
   o1     A      Ana 2026-01-02
   o2     A      Ana 2026-01-03
   o3     A      Ana 2026-01-04
   o4     A      Ben 2026-01-02
   o5     B      Cai 2026-01-02
   o6     B      Cai 2026-01-05


### Pause and predict
1. Calculate each store's average orders per customer.
2. Calculate the share of its customers with at least two orders.
3. Do these measures rank the stores in the same way?
4. Could a newly acquired customer be less likely to qualify simply because they joined late?

Use `talk-it-through`, if available, to discuss the choices one question at a time. Without the skill, ask an agent: “Help me define this metric before implementing it. Ask one question at a time about the decision, population, denominator, time window, and assumptions. Ask me to predict a tiny example before recommending a definition. Summarize our agreement before coding.”

### Worked definition
For this exercise, use **January repeat-customer share**: within each store, the number of customers with at least two completed January orders divided by all customers with at least one completed January order.

A: average orders = 4/2 = **2**; repeat share = 1/2 = **50%**. B: average orders = 2/1 = **2**; repeat share = 1/1 = **100%**.

The metrics answer different questions. This monthly measure gives customers unequal time to return and has a very small denominator at B. A cohort metric with a fixed follow-up window would answer a different question.

In [3]:
per_customer = (orders.groupby(["store", "customer"])
                .size().reset_index(name="n_orders"))
print(per_customer.to_string(index=False))

store customer  n_orders
    A      Ana         3
    A      Ben         1
    B      Cai         2


### Pause and trace the incorrect code
The following code runs. What does its denominator count? Predict both outputs.

In [4]:
repeat = per_customer[per_customer["n_orders"] >= 2]
wrong_share = repeat.groupby("store").size() / orders.groupby("store").size()
print(wrong_share)

store
A    0.25
B    0.50
dtype: float64


### Worked correction
The denominator counts orders, but the definition requires customers: A becomes 25% and B becomes about 33.3%. Compute one Boolean per customer and average those Booleans within each store. True counts as 1 and False as 0.

In [5]:
def repeat_customer_share(order_rows):
    counts = (order_rows.groupby(["store", "customer"])
              .size().reset_index(name="n_orders"))
    counts["repeat"] = counts["n_orders"] >= 2
    return counts.groupby("store")["repeat"].mean()

correct_share = repeat_customer_share(orders)
print(correct_share)
assert correct_share.to_dict() == {"A": 0.5, "B": 1.0}

store
A    0.5
B    1.0
Name: repeat, dtype: float64


### New check before running
Add store C with two customers who each ordered once. Its result must be **0%**, not a missing store. A store with no orders at all has no eligible customers, so its rate is undefined; it would need a separate store roster to appear.

In [6]:
extra_orders = pd.DataFrame({
    "order": ["o7", "o8"], "store": ["C", "C"],
    "customer": ["Dee", "Eli"], "date": ["2026-01-06", "2026-01-07"]
})
extended_orders = pd.concat([orders, extra_orders], ignore_index=True)
extended_share = repeat_customer_share(extended_orders)
print(extended_share)
assert extended_share.loc["C"] == 0

# Independent hand-count check of the original tiny dataset.
hand_share = pd.Series({"A": 1 / 2, "B": 1 / 1})
assert correct_share.to_dict() == hand_share.to_dict()

store
A    0.5
B    1.0
C    0.0
Name: repeat, dtype: float64


**Audit finding:** the original calculation divided repeat customers by orders. Request a customer-level denominator and a test retaining stores with zero repeat customers. Also use a precise label rather than simply “loyalty.”

## 2 · Average tip per visit
**Requirement:** report average tip per visit. A blank tip explicitly means the customer left no tip. Each visit has at least one service.

**Pause and predict:** Visit A has a \$10 tip and two services; visit B has no tip and one service. What is the mean tip per visit? What happens if we calculate the mean after the merge?

In [7]:
visits = pd.DataFrame({"visit": ["A", "B"], "tip": [10.0, None]})
services = pd.DataFrame({"visit": ["A", "A", "B"],
                         "service": ["haircut", "coloring", "haircut"]})
print("Visits:\n", visits.to_string(index=False))
print("\nServices:\n", services.to_string(index=False))

Visits:
 visit  tip
    A 10.0
    B  NaN

Services:
 visit  service
    A  haircut
    A coloring
    B  haircut


In [8]:
clean_visits = visits.copy()
clean_visits["tip"] = clean_visits["tip"].fillna(0)
merged = services.merge(clean_visits, on="visit", validate="many_to_one")
print(merged.to_string(index=False))
wrong_mean_tip = merged["tip"].mean()
print("Mean after merge:", round(wrong_mean_tip, 2))

visit  service  tip
    A  haircut 10.0
    A coloring 10.0
    B  haircut  0.0
Mean after merge: 6.67


### Worked solution
The expected result is **\$5 = (\$10 + \$0) / 2 visits**. The merged table has one row per service. It averages `[10, 10, 0]`, producing about **\$6.67**. The merge validation passes because the join relationship is valid; it does not guarantee that a later aggregation uses the right unit.

In [9]:
mean_tip = clean_visits["tip"].mean()
print("Mean per visit:", mean_tip)
assert mean_tip == 5.0

# A separate hand calculation from the two visits.
assert mean_tip == (10 + 0) / 2

Mean per visit: 5.0


### New check and assumption check
Add a third service to visit A without changing its tip. The mean tip per visit must remain \$5. Then consider a different data dictionary: a blank means **not recorded**. Would filling with zero still be justified?

In [10]:
more_services = pd.concat([
    services, pd.DataFrame({"visit": ["A"], "service": ["styling"]})
], ignore_index=True)
more_merged = more_services.merge(clean_visits, on="visit", validate="many_to_one")
print("Incorrect mean with another service:", more_merged["tip"].mean())
print("Correct visit mean is unchanged:", clean_visits["tip"].mean())
assert clean_visits["tip"].mean() == 5.0

Incorrect mean with another service: 7.5
Correct visit mean is unchanged: 5.0


**If blank means unknown:** the all-visit mean is not identifiable from these data. The mean among recorded tips is \$10, but it must be labeled as such and accompanied by coverage (1 of 2 visits). Neither silently filling zero nor silently excluding the blank establishes the all-visit mean.

**Audit finding:** aggregate tips at the visit level. Preserve the meaning of missing data and verify that adding a service does not change a visit-based metric.

## 3 · Delivery performance
**Request:** “Show our overall on-time delivery rate.”

For the numeric example, each total counts completed deliveries with known outcomes, and each delivery belongs to exactly one store. The on-time rule is already fixed. Later, we will question those assumptions.

In [11]:
delivery = pd.DataFrame({"store": ["A", "B"],
                         "on_time": [9, 1], "total": [10, 2]})
print(delivery.to_string(index=False))

store  on_time  total
    A        9     10
    B        1      2


### Pause and predict
1. What is each store's on-time rate?
2. What is the average of those two rates?
3. What share of all deliveries were on time?
4. Which result gives each store equal weight? Which gives each delivery equal weight?

In [12]:
delivery["rate"] = delivery["on_time"] / delivery["total"]
average_store_rate = delivery["rate"].mean()
overall_delivery_rate = delivery["on_time"].sum() / delivery["total"].sum()
print(delivery.to_string(index=False))
print(f"Average store rate: {average_store_rate:.2%}")
print(f"Overall delivery rate: {overall_delivery_rate:.2%}")

store  on_time  total  rate
    A        9     10   0.9
    B        1      2   0.5
Average store rate: 70.00%
Overall delivery rate: 83.33%


### Worked solution
A is 90%; B is 50%. Their equally weighted average is **70%**. Across all deliveries, **10/12 = 83.33%** were on time. Neither expression is universally wrong: they answer different questions. For the agreed overall share of deliveries, use the ratio of totals.

An independent route is to reconstruct the twelve outcomes and take their mean.

In [13]:
outcomes = pd.Series([True] * 9 + [False] + [True, False])
print(f"Check using individual deliveries: {outcomes.mean():.2%}")
assert abs(outcomes.mean() - overall_delivery_rate) < 1e-12
weighted_rate = (delivery["rate"] * delivery["total"]).sum() / delivery["total"].sum()
assert abs(weighted_rate - overall_delivery_rate) < 1e-12

Check using individual deliveries: 83.33%


### New check before running
Suppose A's records are split into two batches, each with 5 deliveries: one with 5 on-time deliveries and one with 4. B is unchanged. Does the overall delivery rate change? Should it?

In [14]:
batches = pd.DataFrame({"batch": ["A1", "A2", "B"],
                       "on_time": [5, 4, 1], "total": [5, 5, 2]})
batch_mean = (batches["on_time"] / batches["total"]).mean()
batch_overall = batches["on_time"].sum() / batches["total"].sum()
print(f"Unweighted average across batches: {batch_mean:.2%}")
print(f"Overall delivery rate: {batch_overall:.2%}")
assert abs(batch_overall - overall_delivery_rate) < 1e-12

Unweighted average across batches: 76.67%
Overall delivery rate: 83.33%


### Talk it through
Ask one question at a time: Does “on time” mean by the promised timestamp or by the end of the day? Which deliveries are eligible? What does a missing timestamp mean? Is comparing a store with 2 deliveries to one with 1,000 enough to rank performance confidently?

A group with no eligible deliveries has an undefined rate, not automatically 0%. If the entire denominator is zero, report unavailable rather than a fabricated percentage.

**Audit finding:** for an overall delivery share, aggregate counts before dividing. Record the eligibility and on-time rules, and show denominators alongside rates.

## 4 · A simplified weighted lottery
This is a hypothetical lottery, not a model of current immigration policy.

Person A holds 1 entry and person B holds 4 entries. Shuffle all entries uniformly. Scan the line, keeping only each person's first entry, until the available slots are filled. A person can win at most once. Slots never exceed the number of people.

An app uses `slots × person's entries / total entries` as the person's probability of selection.

### Pause and predict
With one slot, what are the two probabilities? With two slots? Can a probability exceed 1?

In [15]:
def linear_odds(weights, slots):
    return [slots * weight / sum(weights) for weight in weights]

print("One slot:", linear_odds([1, 4], 1))
print("Two slots:", linear_odds([1, 4], 2))

One slot: [0.2, 0.8]
Two slots: [0.4, 1.6]


### Worked hand cases
One slot: A = **0.2**, B = **0.8**. Two slots: **both are 1**. The formula gives 0.4 and 1.6 for two slots.

The formula is the expected number of wins under repeated independent entry draws with replacement; that is not a person's probability of selection in this process. Clipping its outputs at 1 still gives 0.4 for A, which is wrong.

### Independent simulation
Read each line: build the entry list; shuffle; keep a person only once; count wins over many runs. A seed makes the sequence reproducible. Simulation estimates probabilities and has sampling error.

In [16]:
def simulate_chances(weights, slots, runs=5000, seed=576):
    # This teaching function accepts positive integer entry counts.
    if not weights or any(type(w) is not int or w <= 0 for w in weights):
        raise ValueError("Use positive integer entry counts.")
    if type(slots) is not int or not 0 <= slots <= len(weights):
        raise ValueError("Slots must be between zero and the number of people.")
    if type(runs) is not int or runs <= 0:
        raise ValueError("Runs must be a positive integer.")
    entries = []
    for person, weight in enumerate(weights):
        entries.extend([person] * weight)
    rng = random.Random(seed)
    wins = [0] * len(weights)
    for _ in range(runs):
        line = entries.copy()
        rng.shuffle(line)
        unique_people = []
        for person in line:
            if person not in unique_people:
                unique_people.append(person)
        for person in unique_people[:slots]:
            wins[person] += 1
    return [count / runs for count in wins]

print("Simulation, one slot:", simulate_chances([1, 4], 1))
print("Simulation, two slots:", simulate_chances([1, 4], 2))

Simulation, one slot: [0.2098, 0.7902]
Simulation, two slots: [1.0, 1.0]


### Verify the correction and its limits
Check zero slots, enough slots for everyone, and three equally weighted people competing for one slot. Probabilities should be in [0, 1], and their sum should equal the number of slots because each run selects exactly that many distinct people.

Agreement with hand cases supports the simulation. It does not prove the process matches a real-world policy.

In [17]:
assert simulate_chances([1, 4], 0) == [0, 0]
assert simulate_chances([1, 4], 2) == [1, 1]
equal = simulate_chances([1, 1, 1], 1)
print("Equal weights, one slot:", equal)
assert all(0 <= chance <= 1 for chance in equal)
assert abs(sum(equal) - 1) < 1e-12
print("Hand-case and structural checks passed.")

Equal weights, one slot: [0.333, 0.3226, 0.3444]
Hand-case and structural checks passed.


### Optional exact check for this tiny lottery
There are only 120 permutations of five distinctly labeled tickets. Enumerating them gives an exact answer and avoids simulation error. This is practical only for very small cases.

In [18]:
tickets = [("A", 1), ("B", 1), ("B", 2), ("B", 3), ("B", 4)]
def exact_chances(slots):
    wins = {"A": 0, "B": 0}
    total = 0
    for line in permutations(tickets):
        people = []
        for person, ticket_id in line:
            if person not in people:
                people.append(person)
        for person in people[:slots]:
            wins[person] += 1
        total += 1
    return {person: count / total for person, count in wins.items()}

print("Exact, one slot:", exact_chances(1))
print("Exact, two slots:", exact_chances(2))
assert exact_chances(1) == {"A": 0.2, "B": 0.8}
assert exact_chances(2) == {"A": 1.0, "B": 1.0}

Exact, one slot: {'A': 0.2, 'B': 0.8}
Exact, two slots: {'A': 1.0, 'B': 1.0}


## Exit reflection and project connection
Choose one number in your team's app. In your own words:
1. What exactly should it measure?
2. What tiny case would help you check it?
3. Which assumption or line deserves inspection?
4. What evidence would support a change request?

The homework is an **individual audit of the shared team app**. Audit at least one feature and one number, document your evidence in `my-audit.md`, and add findings to `change-requests.md`. See the separate homework document for full instructions.

**Remember:** an agent can help implement and check a definition. Your responsibility is to decide whether the definition and evidence answer the business question.